LinkedIn Ground Truth Data Sampler
----------------------
This script reads a CSV file containing scraped LinkedIn posts and generates a random subsets for the ground truth annotation, based on the user-defined size. It filters for English posts so the annotator can understand the text.

In [1]:
import pandas as pd
import os
from langdetect import detect, DetectorFactory
from langdetect.lang_detect_exception import LangDetectException

configuration:

In [ ]:
CONFIG = {
    'INPUT_DIR': '../Data Collection/LinkedIn',
    'INPUT_FILE': 'LinkedIn 2025-26 Dataset - rdbl.csv',
    'OUTPUT_DIR': './Samples',
    'OUTPUT_FILE': 'LinkedIn Ground Truth Sample - rdbl.csv',
    'SAMPLE_SIZE': 30,
    'CSV_SEP': ','
}

DetectorFactory.seed = 42

main script:

In [ ]:
def is_english(text):
    """
    Detects if a string is English.
    Returns True if English, False otherwise.
    """
    if pd.isna(text) or str(text).strip() == "":
        return False
    try:
        # Detect the language of the text
        return detect(text) == 'en'
    except LangDetectException:
        # If detection fails (e.g., only numbers or symbols), include it as well
        return True

def create_linkedin_samples():
    """
    Main function to load data and export random sample.
    """
    
    # Construct full file paths
    input_path = os.path.join(CONFIG['INPUT_DIR'], CONFIG['INPUT_FILE'])
    output_path = os.path.join(CONFIG['OUTPUT_DIR'], CONFIG['OUTPUT_FILE'])

    # Ensure the output directory exists before saving
    if not os.path.exists(CONFIG['OUTPUT_DIR']):
        os.makedirs(CONFIG['OUTPUT_DIR'])
        print(f"Created directory: {CONFIG['OUTPUT_DIR']}")

    try:
        # Load the source CSV file
        print(f"Loading file: {input_path}...")
        df = pd.read_csv(input_path, sep=CONFIG['CSV_SEP'], encoding="utf-8-sig")
        
        total_rows = len(df)
        print(f"Successfully loaded {total_rows} rows.")


        # --- LANGUAGE FILTERING ---
        print("Filtering for English posts... This may take a moment.")
        # Apply the detection function to the 'Text' column
        df = df[df['Text'].apply(is_english)]
        print(f"Filtering complete. {len(df)} English posts remaining.")

        # Generate the random sample
        sample_size = CONFIG['SAMPLE_SIZE']
        df = df.sample(n=sample_size)
            
        # Save results back to CSV
        df.to_csv(output_path, index=False, encoding="utf-8-sig")

        print("-" * 30)
        print("Execution successful:")
        print(f"-> Sample saved: {output_path} ({len(df)} rows)")
        print("-" * 30)

    except FileNotFoundError:
        print(f"Error: The file '{input_path}' was not found. Please check your config.")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")

if __name__ == "__main__":
    create_linkedin_samples()